Is it worth carrying forward each team's Elo from one season to the next? Maybe this makes the model require less adaptation, compared to starting afresh each season.

Our dataset is 10 seasons' results.

In [ ]:
from datasets import SEASONS
SEASONS

Our input data structure is a match history containing all seasons' full time results for one league, for example:

In [ ]:
from datasets import premier, championship, league1, league2, national
premier

Our workhorse function is `compute_elo_history()`, which takes
* a match history
* start and end dates
* a starting elo (default 1400) for teams we haven't see before
* an optional dictionary of known starting elos in case we know them

It returns an 'elo history' structure, which is a list of date, team and new elo.  For example

In [ ]:
from elo_history import compute_elo_history
week1_prem = compute_elo_history(premier, '2015-08-08', '2015-08-15')
week1_prem

For example, there was one game on 2015-08-10, where Manchester City beat West Bromwich Albion and won 20 elo points.

The mean absolute difference between start and end elo gives a crude measure of how much 'work' the algorithm did to get to the final elo.  If the start elos we gave were close to the real strengths of the teams, we'd expect this metric to be lower than if everyone started with the default value.

In [ ]:
def elo_change_mean_abs(elo_history):
    changes = (
        elo_history
        .groupby('Team')['Elo']
        .agg(lambda elos: elos.iloc[-1] - elos.iloc[0])
    )
    return changes.abs().mean()

For a league, we will measure this metric under three regimes:
* reset all teams to 1400 at the start of each season
* elos are carried forward for one season only
* elos are carried forward for all seasons

In [ ]:
import pandas as pd

from elo_history import compute_elo_history, get_final_elos
from datasets import SEASONS

def compute_mean_abs_changes(match_history):

    results = []

    roll_forward_elos = None    # carry forward forever
    previous_reset_elos = None  # carry forward from one year only
    

    for season in SEASONS:
        # choose 1 Aug as cutoff date between seasons
        start_date = pd.Timestamp(f'{season[:4]}-08-01')
        end_date = pd.Timestamp(f'{int(season[:4]) + 1}-08-01')

        # Reset
        reset_history = compute_elo_history(
            match_history,
            start_date,
            end_date,
        )

        # Full roll-forward
        roll_forward_history = compute_elo_history(
            match_history,
            start_date,
            end_date,
            starting_elos=roll_forward_elos,
        )

        # One-season roll-forward
        previous_season_history = compute_elo_history(
            match_history,
            start_date,
            end_date,
            starting_elos=previous_reset_elos,
        )

        results.append({
            'season': season,
            'reset': elo_change_mean_abs(reset_history),
            'roll_forward': elo_change_mean_abs(roll_forward_history),
            'previous_season': elo_change_mean_abs(previous_season_history),
        })

        roll_forward_elos = get_final_elos(roll_forward_history)
        previous_reset_elos = get_final_elos(reset_history)

    return pd.DataFrame(results)

In [ ]:
import matplotlib.pyplot as plt

def plot_mean_abs_change_sequence(league_name, seq):
    
    seq.plot(
        x='season',
        y=['roll_forward', 'previous_season', 'reset'],
        kind='bar',
        figsize=(12, 6),
    )

    plt.ylabel('Mean absolute Elo change')
    plt.xlabel('Season')
    plt.title(league_name)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

For the Premier league, rolling elos forward one year appears to be a win over resetting each year.  Rolling forward indefinitely gives a further advantage some years.

In [ ]:
prem_seq = compute_mean_abs_changes(premier)
plot_mean_abs_change_sequence('Premier league', prem_seq)

Unfortunately, there is no such effect for the National league, and rolling forward elos is no better, sometimes worse.

In [ ]:
national_seq = compute_mean_abs_changes(national)
plot_mean_abs_change_sequence('National', national_seq)

Conclude that the strength of teams as measured by Elo persists across season boundaries in the Premier league, but not in the National league.